# RAG — Retrieval-Augmented Generation
### A Beginner-Friendly, End-to-End Lab Notebook

Welcome! In this notebook you will build a **complete RAG pipeline from scratch**, step by step, using simple and well-known open-source tools.

## What is RAG?

**Retrieval-Augmented Generation (RAG)** is an architecture that combines:

1. A **retrieval system** — finds relevant pieces of information from a knowledge base.
2. A **generative LLM** — writes a natural-language answer using that retrieved information.

Instead of asking an LLM a question and hoping it "remembers" the right answer from training, we **first fetch the relevant facts**, then **ask the LLM to answer using those facts only**.

## Why do we need RAG?

A plain LLM has three important limitations:

| Limitation | Explanation |
|---|---|
| **Frozen knowledge** | The LLM only knows what existed in its training data, up to a cutoff date. |
| **No private/domain data** | The LLM has never seen your company's documents, PDFs, or internal wikis. |
| **Hallucination** | When the LLM doesn't know an answer, it may still generate a confident-sounding but *incorrect* answer. |

RAG fixes all three problems by **grounding** the LLM's answer in real, retrieved documents.

## RAG Architecture (high level)

```text
Documents → Chunking → Embeddings → Vector Database → Retrieval → Context → LLM → Final Answer
```

### Mermaid Diagram

```mermaid
flowchart TD
    A[Documents] --> B[Chunking]
    B --> C[Embedding Model]
    C --> D[(Vector Database)]
    Q[User Question] --> QE[Query Embedding]
    QE --> D
    D --> R[Top-K Retrieved Chunks]
    R --> CTX[Context Construction]
    CTX --> P[Prompt + Question]
    P --> LLM[LLM]
    LLM --> ANS[Final Grounded Answer]
```

## Key Terminology

| Component | Role |
|---|---|
| **LLM (Large Language Model)** | Generates natural-language text; the "writer" of the final answer. |
| **Embedding Model** | Converts text into a numerical vector that captures its meaning. |
| **Vector Database** | Stores embeddings and performs fast similarity search. |
| **Retriever** | The logic that queries the Vector Database and returns the most relevant chunks. |
| **Generator** | The LLM call that turns (question + retrieved context) into a final answer. |

> **Important distinction:** RAG is the *complete architecture*. The Vector Database is only *one component* inside RAG — not RAG itself.

---
### Notebook Roadmap

```text
01 — Introduction
02 — Setup
03 — Documents
04 — Chunking
05 — Embeddings
06 — Vector Database
07 — Retrieval
08 — Context
09 — Prompt
10 — Generation
11 — Complete RAG Pipeline
12 — Evaluation
13 — Final Architecture
```


## 02 — Environment Setup

This lab runs **fully locally — no paid API, no API key required**. Every model (embedding + generation) downloads once from Hugging Face and then runs on your own machine (CPU is fine).

We keep dependencies minimal and beginner-friendly:

- `langchain-text-splitters` — text splitting utilities
- `sentence-transformers` — lightweight local embedding model
- `transformers` — lightweight local instruction-following model for generation (no API needed)
- `chromadb` — simple local vector database
- `pypdf` — in case you want to load PDF documents later

> Want to swap in a hosted LLM (OpenAI, Anthropic, Groq, ...) instead of the local model? See the note at the end of Section 12 — the rest of the pipeline stays identical either way.


In [ ]:
%pip install -q langchain-text-splitters sentence-transformers transformers chromadb pypdf


In [ ]:
import os
import glob
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

print("✅ Setup complete — no API key needed, everything below runs locally.")


✅ Setup complete — no API key needed, everything below runs locally.


## 03 — Dataset / Documents

```text
Document
   ↓
Raw Text
```

We use a small, realistic local knowledge base: five short `.txt` files covering **AI, Machine Learning, Deep Learning, Generative AI, and RAG**.

This keeps the lab fully reproducible and runnable offline (no external downloads needed).


In [ ]:
# Sample knowledge-base documents.
# In a real project these would already exist as files; here we create them
# programmatically so the notebook is fully self-contained and reproducible.

DOCS_CONTENT = {
    "ai.txt": """Artificial Intelligence (AI)

Artificial Intelligence is the branch of computer science focused on building systems that can perform tasks that normally require human intelligence. These tasks include understanding language, recognizing images, making decisions, and solving problems.

AI systems can be rule-based, where humans explicitly program the logic, or data-driven, where the system learns patterns from data. Modern AI applications include voice assistants, recommendation systems, autonomous vehicles, and medical diagnosis tools.

AI is often divided into three categories: Narrow AI (systems designed for a specific task), General AI (systems with human-level reasoning across many tasks, still theoretical), and Super AI (a hypothetical AI surpassing human intelligence). Today, almost all deployed AI systems are Narrow AI.

The field of AI includes many sub-disciplines such as Machine Learning, Natural Language Processing, Computer Vision, Robotics, and Expert Systems. Each sub-discipline focuses on a different aspect of simulating intelligent behavior.""",

    "ml.txt": """Machine Learning (ML)

Machine Learning is a subset of Artificial Intelligence that enables computers to learn patterns from data without being explicitly programmed for every rule. Instead of writing fixed instructions, we feed the system examples, and it learns a function that maps inputs to outputs.

There are three main types of Machine Learning: Supervised Learning, where the model learns from labeled examples (input-output pairs); Unsupervised Learning, where the model finds patterns or clusters in unlabeled data; and Reinforcement Learning, where an agent learns by interacting with an environment and receiving rewards or penalties.

Common Machine Learning algorithms include Linear Regression, Logistic Regression, Decision Trees, Random Forests, Support Vector Machines, and k-Nearest Neighbors. These algorithms are often used for tasks like classification, regression, and clustering.

A typical Machine Learning workflow includes: collecting data, cleaning and preprocessing it, splitting it into training and testing sets, selecting a model, training the model, evaluating its performance, and finally deploying it into production.""",

    "dl.txt": """Deep Learning (DL)

Deep Learning is a specialized subset of Machine Learning based on artificial neural networks with many layers, hence the term "deep". These layered networks are capable of automatically learning hierarchical representations of data, from simple features to complex patterns.

A neural network consists of layers of interconnected nodes called neurons. Each connection has a weight that is adjusted during training using an algorithm called backpropagation, which minimizes the difference between the predicted output and the actual output.

Deep Learning has achieved remarkable success in fields such as computer vision (using Convolutional Neural Networks, or CNNs), natural language processing (using Recurrent Neural Networks, Long Short-Term Memory networks, and more recently Transformers), and speech recognition.

Deep Learning models typically require large amounts of data and significant computational power, often using GPUs or TPUs, to train effectively. This is one of the key differences between traditional Machine Learning and Deep Learning approaches.""",

    "genai.txt": """Generative AI (GenAI)

Generative AI refers to a category of Artificial Intelligence models that can create new content, such as text, images, audio, code, or video, rather than simply analyzing or classifying existing data. These models learn the underlying patterns and structure of their training data and use that knowledge to generate novel outputs.

The most well-known type of Generative AI today is the Large Language Model (LLM), such as GPT-based models. LLMs are trained on massive amounts of text data and learn to predict the next word in a sequence, which allows them to generate coherent and contextually relevant text.

Other forms of Generative AI include image generation models like Diffusion Models and Generative Adversarial Networks (GANs), which can create realistic images from text descriptions or random noise.

One major limitation of Generative AI, especially LLMs, is a phenomenon called "hallucination", where the model generates information that sounds plausible but is factually incorrect. This happens because LLMs generate text based on learned patterns rather than verified facts, and they have no built-in way to check their answers against a reliable source of truth.""",

    "rag.txt": """Retrieval-Augmented Generation (RAG)

Retrieval-Augmented Generation, or RAG, is an architecture that combines a retrieval system with a generative Large Language Model to produce answers that are grounded in external, up-to-date, or domain-specific knowledge. Instead of relying only on what the LLM memorized during training, RAG retrieves relevant information from an external knowledge source at query time and provides it to the LLM as context.

The core problem RAG solves is that LLMs alone have three major limitations: their knowledge is frozen at the time of training, they cannot access private or domain-specific documents, and they can hallucinate when they lack the correct information. RAG addresses all three problems by injecting fresh, relevant, and verifiable information directly into the prompt.

A typical RAG pipeline works as follows: documents are split into smaller chunks, each chunk is converted into a numerical vector using an embedding model, and these vectors are stored in a Vector Database. When a user asks a question, the question itself is converted into a vector, and the Vector Database performs a similarity search to find the most relevant chunks. These chunks are combined into a context, which is passed to the LLM along with the original question, and the LLM generates a final answer grounded in that context.

RAG is widely used in enterprise chatbots, customer support systems, legal and medical document search, and any application where answers need to be accurate, traceable, and based on a specific set of documents rather than general world knowledge.""",
}

for filename, content in DOCS_CONTENT.items():
    with open(os.path.join(DATA_DIR, filename), "w", encoding="utf-8") as f:
        f.write(content)

print(f"✅ Created {len(DOCS_CONTENT)} documents in '{DATA_DIR}/'")


✅ Created 5 documents in 'data/'


## 04 — Document Loading

A clean document loader should tell us, at a glance:

- How many documents were loaded
- Their names
- Their size (in characters)
- A short content preview (never the full document — keep outputs readable)


In [ ]:
def load_documents(data_dir: str) -> list[dict]:
    """
    Load every .txt file in `data_dir` into a simple list of dicts.
    Each dict has: source (filename), text (raw content).
    """
    documents = []
    filepaths = sorted(glob.glob(os.path.join(data_dir, "*.txt")))

    if not filepaths:
        raise FileNotFoundError(f"No .txt documents found in '{data_dir}'.")

    for path in filepaths:
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        documents.append({"source": os.path.basename(path), "text": text})

    return documents


documents = load_documents(DATA_DIR)

print(f"Number of documents loaded: {len(documents)}\n")
print(f"{'Document':<15}{'Characters':<12}Preview")
print("-" * 70)
for doc in documents:
    preview = doc["text"].replace("\n", " ")[:50] + "..."
    print(f"{doc['source']:<15}{len(doc['text']):<12}{preview}")


Number of documents loaded: 5

Document       Characters  Preview
----------------------------------------------------------------------
ai.txt         1073        Artificial Intelligence (AI)  Artificial Intellige...
dl.txt         1090        Deep Learning (DL)  Deep Learning is a specialized...
genai.txt      1204        Generative AI (GenAI)  Generative AI refers to a c...
ml.txt         1152        Machine Learning (ML)  Machine Learning is a subse...
rag.txt        1603        Retrieval-Augmented Generation (RAG)  Retrieval-Au...


## 05 — Text Chunking

### Why do we need chunking?

Documents are often too long to embed and retrieve efficiently as a single unit. If we embed an entire document as one vector, we lose fine-grained detail — a similarity search would only tell us "this whole document is somewhat relevant", not "this exact paragraph answers your question".

```text
Large Document
      ↓
Smaller Chunks
      ↓
Better Retrieval
```

By splitting documents into smaller, overlapping **chunks**, we allow the retriever to pinpoint the most relevant *piece* of text, not just the most relevant *document*.

### Key parameters

| Parameter | Meaning |
|---|---|
| `chunk_size` | Maximum number of characters in each chunk. |
| `chunk_overlap` | Number of characters shared between consecutive chunks, so we don't cut a sentence's meaning in half at the boundary. |

We use a **Recursive Character Text Splitter**, which tries to split on paragraph breaks first, then sentences, then words — keeping chunks as semantically coherent as possible.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = 500
chunk_overlap = 50

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", ". ", " ", ""],
)


def chunk_documents(documents: list[dict]) -> list[dict]:
    """
    Split every document's text into overlapping chunks.
    Each chunk keeps track of its source document for later attribution.
    """
    chunks = []
    for doc in documents:
        splits = text_splitter.split_text(doc["text"])
        for i, chunk_text in enumerate(splits):
            chunks.append({
                "id": f"{doc['source']}_chunk{i}",
                "text": chunk_text,
                "source": doc["source"],
            })
    return chunks


chunks = chunk_documents(documents)

print(f"Total number of chunks created: {len(chunks)}\n")
print("Example chunk:")
print("-" * 70)
print(f"ID:     {chunks[0]['id']}")
print(f"Source: {chunks[0]['source']}")
print(f"Length: {len(chunks[0]['text'])} characters")
print(f"Text:   {chunks[0]['text'][:200]}...")


Total number of chunks created: 20

Example chunk:
----------------------------------------------------------------------
ID:     ai.txt_chunk0
Source: ai.txt
Length: 284 characters
Text:   Artificial Intelligence (AI)

Artificial Intelligence is the branch of computer science focused on building systems that can perform tasks that normally require human intelligence. These tasks include...


## 06 — Embeddings

> An **embedding** converts text into a numerical vector that captures its semantic meaning.

Texts with similar meaning end up with vectors that are close together in this high-dimensional space, even if they don't share the exact same words. This is what makes **semantic search** possible — we're searching by *meaning*, not by keyword matching.

We use a lightweight, fast, and free local Sentence Transformer model: **`all-MiniLM-L6-v2`**. It produces 384-dimensional vectors and runs comfortably on CPU.


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Demonstrate embedding a single chunk
example_embedding = embedding_model.encode(chunks[0]["text"])

print(f"Vector dimension: {example_embedding.shape[0]}")
print(f"First 8 dimensions of the example vector:\n{example_embedding[:8]}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector dimension: 384
First 8 dimensions of the example vector:
[-0.02224956 -0.00400587  0.00826754  0.02169827 -0.00039299 -0.0959276
  0.05029861  0.02081897]


In [ ]:
# Embed ALL chunks at once (batched, efficient)
chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embedding_model.encode(chunk_texts, show_progress_bar=True)

print(f"\n✅ Generated {len(chunk_embeddings)} embeddings, "
      f"each with {chunk_embeddings.shape[1]} dimensions.")

# Why are vectors useful for semantic search?
print("""
Why vectors matter:
- Two chunks with similar MEANING will have vectors that are close together,
  even if they use completely different words.
- This lets us search by "meaning" instead of by exact keyword matching.
""")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Generated 20 embeddings, each with 384 dimensions.

Why vectors matter:
- Two chunks with similar MEANING will have vectors that are close together,
  even if they use completely different words.
- This lets us search by "meaning" instead of by exact keyword matching.



## 07 — Vector Database

A **Vector Database** stores embeddings alongside their original text and metadata, and provides fast **similarity search**: given a query vector, it returns the most similar stored vectors.

We use **ChromaDB**, a simple, local, open-source vector database — perfect for learning and small projects.

> **Important distinction:**
> ```text
> RAG = Complete Architecture
> Vector Database = One Component Inside RAG
> ```
> The Vector Database only handles storage + similarity search. Chunking, embedding, prompting, and generation are separate components that RAG combines together.


In [ ]:
import chromadb

# In-memory Chroma client (data lives only for this notebook session).
# Use chromadb.PersistentClient(path="chroma_db") if you want it saved to disk.
chroma_client = chromadb.EphemeralClient()

# Start fresh if the collection already exists (e.g. re-running the notebook)
try:
    chroma_client.delete_collection("rag_lab_collection")
except Exception:
    pass

collection = chroma_client.create_collection(name="rag_lab_collection")

collection.add(
    ids=[c["id"] for c in chunks],
    embeddings=[emb.tolist() for emb in chunk_embeddings],
    documents=[c["text"] for c in chunks],
    metadatas=[{"source": c["source"]} for c in chunks],
)

print(f"✅ Stored {collection.count()} chunks in the Vector Database.")


✅ Stored 20 chunks in the Vector Database.


## 08 — Retrieval

Retrieval is the process of turning a user's question into the same vector space as our documents, then asking the Vector Database for the closest matches.

```text
User Query
    ↓
Query Embedding
    ↓
Similarity Search
    ↓
Top-K Chunks
```

We use `k = 3` — meaning we retrieve the 3 most relevant chunks for every question.


In [ ]:
k = 3


def retrieve(query: str, k: int = 3) -> list[dict]:
    """
    Embed the query and perform a similarity search against the Vector Database.
    Returns the top-k most relevant chunks with their distance and source.
    """
    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
    )

    retrieved = []
    for rank, (doc_text, distance, metadata) in enumerate(
        zip(results["documents"][0], results["distances"][0], results["metadatas"][0]),
        start=1,
    ):
        retrieved.append({
            "rank": rank,
            "distance": distance,
            "source": metadata["source"],
            "text": doc_text,
        })
    return retrieved


def print_retrieved(retrieved: list[dict]) -> None:
    for r in retrieved:
        preview = r["text"].replace("\n", " ")[:120] + "..."
        print(f"[Rank {r['rank']}] source={r['source']}  distance={r['distance']:.4f}")
        print(f"   {preview}\n")


# Quick sanity check
sample_results = retrieve("What is deep learning?", k=k)
print_retrieved(sample_results)


[Rank 1] source=dl.txt  distance=0.3971
   Deep Learning (DL)  Deep Learning is a specialized subset of Machine Learning based on artificial neural networks with m...

[Rank 2] source=dl.txt  distance=0.7047
   Deep Learning has achieved remarkable success in fields such as computer vision (using Convolutional Neural Networks, or...

[Rank 3] source=dl.txt  distance=0.7363
   Deep Learning models typically require large amounts of data and significant computational power, often using GPUs or TP...



## 09 — Retrieval Experiment

Before connecting anything to an LLM, it's critical to verify that **retrieval itself works well**. If retrieval returns irrelevant chunks, the LLM has no chance of producing a good answer — garbage in, garbage out.

> **Good RAG depends heavily on good retrieval.**

Let's test the retriever with several realistic questions and inspect what comes back.


In [ ]:
test_questions = [
    "What is deep learning?",
    "What is RAG?",
    "Why are embeddings useful?",
    "What is the role of a vector database?",
    "What are the types of machine learning?",
]

for question in test_questions:
    print("=" * 70)
    print(f"QUESTION: {question}")
    print("=" * 70)
    results = retrieve(question, k=k)
    print_retrieved(results)


QUESTION: What is deep learning?
[Rank 1] source=dl.txt  distance=0.3971
   Deep Learning (DL)  Deep Learning is a specialized subset of Machine Learning based on artificial neural networks with m...

[Rank 2] source=dl.txt  distance=0.7047
   Deep Learning has achieved remarkable success in fields such as computer vision (using Convolutional Neural Networks, or...

[Rank 3] source=dl.txt  distance=0.7363
   Deep Learning models typically require large amounts of data and significant computational power, often using GPUs or TP...

QUESTION: What is RAG?
[Rank 1] source=rag.txt  distance=0.5407
   RAG is widely used in enterprise chatbots, customer support systems, legal and medical document search, and any applicat...

[Rank 2] source=rag.txt  distance=1.0259
   A typical RAG pipeline works as follows: documents are split into smaller chunks, each chunk is converted into a numeric...

[Rank 3] source=rag.txt  distance=1.0298
   Retrieval-Augmented Generation (RAG)  Retrieval-Augmented 

### Retrieval Observation

Looking at the results above, the retrieved chunks are generally **topically aligned** with each question — for example, the "deep learning" question pulls chunks from `dl.txt`, and the "vector database" question pulls chunks from `rag.txt`. Lower `distance` values indicate a closer semantic match.

This confirms the retriever is doing its job: finding the *right document and passage*, not just any random text. If a question's top result had a high distance or came from an unrelated document, that would be a sign to reconsider `chunk_size`, `k`, or the embedding model.


## 10 — Context Construction

Once we have the top-k retrieved chunks, we need to combine them into a single block of text — the **context** — that we will hand to the LLM.

```text
Retrieved Documents
        ↓
     Context
        ↓
      LLM
```


In [ ]:
def build_context(retrieved_docs: list[dict]) -> str:
    """
    Combine retrieved chunks into a single, clearly-labeled context string.
    Labeling each chunk with its source enables later source attribution.
    """
    context_parts = []
    for r in retrieved_docs:
        context_parts.append(f"[Source: {r['source']}]\n{r['text']}")
    return "\n\n---\n\n".join(context_parts)


example_context = build_context(retrieve("What is RAG?", k=k))
print(example_context[:600] + "...")


[Source: rag.txt]
RAG is widely used in enterprise chatbots, customer support systems, legal and medical document search, and any application where answers need to be accurate, traceable, and based on a specific set of documents rather than general world knowledge.

---

[Source: rag.txt]
A typical RAG pipeline works as follows: documents are split into smaller chunks, each chunk is converted into a numerical vector using an embedding model, and these vectors are stored in a Vector Database. When a user asks a question, the question itself is converted into a vector, and the Vector Database pe...


## 11 — Prompt Engineering for RAG

The prompt is where we instruct the LLM to **only** use the provided context, and to explicitly admit when it doesn't know the answer. This single instruction is one of the most effective ways to reduce hallucination in a RAG system.


In [ ]:
RAG_PROMPT_TEMPLATE = """You are a helpful AI assistant.

Answer the user's question using ONLY the provided context.

If the answer cannot be found in the context, say:
"I don't have enough information in the provided documents."

Context:
{context}

Question:
{question}

Answer:"""


def build_prompt(question: str, context: str) -> str:
    return RAG_PROMPT_TEMPLATE.format(context=context, question=question)


print(build_prompt("What is RAG?", "[example context would go here]"))


You are a helpful AI assistant.

Answer the user's question using ONLY the provided context.

If the answer cannot be found in the context, say:
"I don't have enough information in the provided documents."

Context:
[example context would go here]

Question:
What is RAG?

Answer:


**Why this instruction helps reduce hallucination:**

By explicitly restricting the LLM to the given context and giving it an explicit, safe "I don't know" fallback phrase, we remove the LLM's incentive to guess. Without this instruction, the LLM will fall back on its general training knowledge — which may be outdated, generic, or simply wrong for your specific documents.


## 12 — Generation

Now we connect the constructed prompt to an actual LLM call. **No API key, no paid service** — we use a small, free, instruction-following model (`google/flan-t5-base`) that downloads once from Hugging Face and runs locally on CPU.

The complete flow so far:

```text
Question
   ↓
Embedding
   ↓
Vector Search
   ↓
Top-K Chunks
   ↓
Context
   ↓
Prompt
   ↓
LLM
   ↓
Answer
```

> **Want a hosted LLM instead?** (OpenAI, Anthropic, Groq, ...) Just replace the body of `generate_answer` with a call to that provider's SDK — everything else in this notebook (retrieval, context, prompt) stays exactly the same. That's the beauty of RAG: retrieval and generation are cleanly separated.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# A small, free, local instruction-following model — no API key needed.
# We load it directly via Auto classes (not the pipeline() shorthand) because
# newer versions of `transformers` don't register "text2text-generation" as a
# pipeline task name for every install — using Auto classes works everywhere.
GEN_MODEL_NAME = "google/flan-t5-base"
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL_NAME)


def generate_answer(question: str, context: str) -> str:
    """
    Run the RAG-grounded prompt through a local LLM and return its answer.
    Includes basic error handling so an unexpected model issue doesn't crash the notebook.
    """
    prompt = build_prompt(question, context)

    try:
        inputs = gen_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        output_ids = gen_model.generate(**inputs, max_new_tokens=200)
        return gen_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    except Exception as e:
        return f"[Local generation failed: {e}]"


# Quick test
demo_context = build_context(retrieve("What is RAG?", k=k))
demo_answer = generate_answer("What is RAG?", demo_context)
print(demo_answer)


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

an architecture that combines a retrieval system with a generative Large Language Model to produce answers that are grounded in external, up-to-date, or domain-specific knowledge


## 13 — Complete RAG Pipeline

We now wrap every step into **one clean function**: `rag_pipeline`. This is the function a real application would call for every incoming user question.

Steps performed:
1. Query embedding
2. Retrieval
3. Context construction
4. Prompt creation
5. LLM generation
6. Return answer + retrieved sources


In [ ]:
def rag_pipeline(question: str, k: int = 3) -> dict:
    """
    Full end-to-end RAG pipeline for a single question.
    Returns a dict with the answer and the list of source documents used.
    """
    retrieved_docs = retrieve(question, k=k)
    context = build_context(retrieved_docs)
    answer = generate_answer(question, context)

    sources = sorted(set(r["source"] for r in retrieved_docs))

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "retrieved_docs": retrieved_docs,
    }


question = "What is Retrieval-Augmented Generation?"
result = rag_pipeline(question, k=k)

print(f"Question: {result['question']}\n")
print(f"Answer:\n{result['answer']}\n")
print(f"Sources: {', '.join(result['sources'])}")


Question: What is Retrieval-Augmented Generation?

Answer:
an architecture that combines a retrieval system with a generative Large Language Model to produce answers that are grounded in external, up-to-date, or domain-specific knowledge

Sources: genai.txt, rag.txt


## 14 — Source Attribution

A key advantage of RAG over a plain LLM is **traceability** — we can always show *which documents* backed up the answer. Let's display the pipeline's output in a clean, professional format.


In [ ]:
def display_rag_result(result: dict) -> None:
    print("### Answer\n")
    print(result["answer"])
    print("\n### Sources\n")
    for src in result["sources"]:
        print(f"- {src}")


display_rag_result(result)


### Answer

an architecture that combines a retrieval system with a generative Large Language Model to produce answers that are grounded in external, up-to-date, or domain-specific knowledge

### Sources

- genai.txt
- rag.txt


## 14 (cont.) — Try It Yourself (Interactive)

Want to ask your own question instead of the hard-coded examples above? Run the cell below — it will prompt you for a question using `input()`, run it through the full `rag_pipeline`, and display the answer with its sources.

> Tip: try a question that IS covered by the 5 documents (e.g. "What is a neural network?"), then try one that ISN'T (e.g. "What's the weather today?") to see the safety fallback in action.


In [ ]:
user_question = input("Ask your question: ")

interactive_result = rag_pipeline(user_question, k=k)
display_rag_result(interactive_result)


Ask your question: What is the role of a vector database?
### Answer

performs a similarity search to find the most relevant chunks

### Sources

- ai.txt
- ml.txt
- rag.txt


## 15 — RAG vs LLM Experiment

Let's directly compare answering **with** and **without** retrieval, using the exact same question.


In [ ]:
comparison_question = "What is Retrieval-Augmented Generation?"

# ---- Without RAG: ask the local LLM directly, no retrieved context at all ----
def ask_llm_directly(question: str) -> str:
    try:
        inputs = gen_tokenizer(question, return_tensors="pt", truncation=True, max_length=512)
        output_ids = gen_model.generate(**inputs, max_new_tokens=200)
        return gen_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
    except Exception as e:
        return f"[Local generation failed: {e}]"


without_rag_answer = ask_llm_directly(comparison_question)
with_rag_result = rag_pipeline(comparison_question, k=k)

print("WITHOUT RAG:\n" + without_rag_answer)
print("\n" + "-" * 70 + "\n")
print("WITH RAG:\n" + with_rag_result["answer"])
print(f"\nSources used: {', '.join(with_rag_result['sources'])}")


WITHOUT RAG:
a genetic variation

----------------------------------------------------------------------

WITH RAG:
an architecture that combines a retrieval system with a generative Large Language Model to produce answers that are grounded in external, up-to-date, or domain-specific knowledge

Sources used: genai.txt, rag.txt


| Approach | External Knowledge | Grounding | Sources |
|---|---|---|---|
| LLM | Limited to model knowledge | Low | No |
| RAG | Retrieved documents | Higher | Yes |

The RAG answer is grounded in our actual documents and comes with source attribution, while the plain LLM answer relies purely on whatever it memorized during training.


## 16 — Failure Case (RAG Safety Mechanism)

A well-built RAG system should **refuse to answer** when the knowledge base simply doesn't contain the information — instead of inventing a plausible-sounding but false answer.

Let's test this with a question that our 5 documents cannot possibly answer.


In [ ]:
failure_question = "What is the population of Mars?"

failure_result = rag_pipeline(failure_question, k=k)
display_rag_result(failure_result)


### Answer

I don't have enough information in the provided documents.

### Sources

- ai.txt
- ml.txt
- rag.txt


**Why this matters:** even though the retriever will still return the *closest* chunks (retrieval always returns *something*, even if nothing is truly relevant), the prompt instruction from Section 11 tells the LLM to say `"I don't have enough information in the provided documents."` rather than hallucinate an answer. This is one of RAG's most important safety mechanisms — it prevents confidently wrong answers on out-of-scope questions.


## 17 — Final Architecture Summary

```text
                    ┌──────────────────┐
                    │   User Query     │
                    └────────┬─────────┘
                             ↓
                    ┌──────────────────┐
                    │ Query Embedding  │
                    └────────┬─────────┘
                             ↓
                    ┌──────────────────┐
                    │  Vector Search   │
                    └────────┬─────────┘
                             ↓
                    ┌──────────────────┐
                    │ Top-K Documents  │
                    └────────┬─────────┘
                             ↓
                    ┌──────────────────┐
                    │ Context + Prompt │
                    └────────┬─────────┘
                             ↓
                    ┌──────────────────┐
                    │       LLM        │
                    └────────┬─────────┘
                             ↓
                    ┌──────────────────┐
                    │ Answer + Sources │
                    └──────────────────┘
```


## 12 (cont.) — Visualizations

A few lightweight, professional visualizations to better understand our pipeline's behavior:

1. Number of chunks per document
2. Retrieved chunk relevance (distance) for a sample query
3. A simple RAG pipeline architecture diagram


In [ ]:
import plotly.express as px
import pandas as pd

# 1. Number of chunks per document
chunk_counts = pd.Series([c["source"] for c in chunks]).value_counts().reset_index()
chunk_counts.columns = ["document", "num_chunks"]

fig1 = px.bar(
    chunk_counts, x="document", y="num_chunks",
    title="Number of Chunks per Document",
    labels={"document": "Document", "num_chunks": "Chunk Count"},
)
fig1.show()


In [ ]:
# 2. Retrieved chunk relevance (distance) for a sample query
sample_query = "What is RAG?"
sample_retrieved = retrieve(sample_query, k=5)

dist_df = pd.DataFrame(sample_retrieved)[["rank", "source", "distance"]]

fig2 = px.bar(
    dist_df, x="rank", y="distance", color="source",
    title=f"Retrieval Distance by Rank — Query: '{sample_query}'",
    labels={"rank": "Rank", "distance": "Distance (lower = more relevant)"},
)
fig2.show()


In [ ]:
# 3. Simple RAG pipeline diagram (Plotly annotated flow)
import plotly.graph_objects as go

stages = ["Query", "Embedding", "Vector Search", "Top-K Chunks", "Context", "Prompt", "LLM", "Answer"]
x_positions = list(range(len(stages)))

fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=x_positions, y=[0] * len(stages),
    mode="markers+text",
    marker=dict(size=28, color="#4C78A8"),
    text=stages, textposition="top center",
))
for i in range(len(stages) - 1):
    fig3.add_annotation(
        x=x_positions[i + 1], y=0, ax=x_positions[i], ay=0,
        xref="x", yref="y", axref="x", ayref="y",
        showarrow=True, arrowhead=3, arrowsize=1.5, arrowwidth=2,
    )

fig3.update_layout(
    title="RAG Pipeline Flow",
    xaxis=dict(visible=False), yaxis=dict(visible=False, range=[-1, 1]),
    height=250, showlegend=False,
)
fig3.show()


## Mini Challenge

Now it's your turn! Extend this notebook by completing the following TODOs:

1. **Add a new document** — create a 6th `.txt` file about a related topic (e.g. "Prompt Engineering" or "Vector Search Algorithms").
2. **Re-index the Vector Database** — re-run the chunking, embedding, and storage cells so the new document is included.
3. **Ask a new question** — pick a question that only the new document can answer, and run it through `rag_pipeline`.
4. **Change `k` from 3 to 5** — rerun the retrieval experiment and observe how the results change.
5. **Compare the retrieved results** — are the extra 2 chunks (rank 4 and 5) still relevant, or does quality drop off?
6. **Explain how retrieval changed** — write a short Markdown note with your observations.

```python
# TODO: Add your new document here
# TODO: Re-run chunk_documents() and the embedding/storage cells
# TODO: result = rag_pipeline("your new question", k=5)
# TODO: Write your observations below
```

---

## Optional Extensions (beyond the scope of this lab)

If you want to go further after mastering the fundamentals, you could explore:

- **Re-rankers** — a second model that re-scores retrieved chunks for higher precision.
- **Hybrid search** — combining keyword (BM25) search with vector search.
- **Multi-query retrieval** — generating multiple reformulations of a question to widen the search.
- **LangGraph / agentic RAG** — adding decision-making loops around retrieval and generation.
- **Formal RAG evaluation frameworks** — e.g. RAGAS, for automated faithfulness and relevance scoring.

These are intentionally **not** covered here — the goal of this notebook is to build a solid, correct understanding of the fundamental RAG pipeline first.

---

### 🎉 Congratulations!

You've built a complete, working RAG pipeline from scratch — you now understand:

**Why RAG exists → How documents become vectors → How retrieval works → How context reaches the LLM → How the final grounded answer is generated.**
